In [11]:
from dotenv import load_dotenv
import os, openai, json

load_dotenv()
client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

messages = []

In [12]:
def get_weather(city):
    return "33 degrees celcius."

FUNCTION_MAP = {
    'get_weather': get_weather
}

In [13]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "A function to get the weather of a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city to get the weather of."
                    }
                },
                "required": ["city"]
            }
        }
    }
]

In [16]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        # Tool Calling
        messages.append({
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [{
                "id": tool_call.id,
                "type": "function",
                "function": {
                    "name": tool_call.function.name,
                    "arguments": tool_call.function.arguments
                }
            } for tool_call in message.tool_calls]
        })

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments
            print(f"Calling function: {function_name} with {arguments}")

            try: 
                arguments = json.loads(arguments) # Json String -> Python Dictionary
            except json.JSONDecodeError:
                arguments = {}
            
            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments) # ** == 'city'='Spain' -> city=Spain

            print(f"Ran {function_name} with args {arguments} for a result of {result}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": result
            })
        call_ai()
    else:
        messages.append({
            "role": "assistant",
            "content": message.content
        })
        print(f"AI: {message}")
    
def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages = messages, 
        tools=TOOLS
    )
    # tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_vYl9nwoor3JesvKQkVSlMKW9', function=Function(arguments='{"city":"Spain"}', name='get_weather')
    process_ai_response(response.choices[0].message)

In [17]:
while True:
    message = input("Send a message to the LLM...")
    if message == 'quit' or message == 'q':
        break
    else:
        messages.append({
            "role": "user",
            "content": message
        })
        print(f"User: {message}")
        call_ai()


User: my name is nico
AI: ChatCompletionMessage(content='Hello Nico! How can I assist you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)
User: what is my name
AI: ChatCompletionMessage(content='Your name is Nico. How can I help you further?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)
User: what is the weather in spain
Calling function: get_weather with {"city":"Spain"}
Ran get_weather with args {'city': 'Spain'} for a result of 33 degrees celcius.
AI: ChatCompletionMessage(content='The weather in Spain is currently 33 degrees Celsius. Would you like to know the weather for a specific city in Spain?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)
